# ShipsEar CNN and ResNet18 — Complete Notebook Implementation

This notebook contains the **entire implementation**: metadata inspection, audio preparation, Dataset A/B splitting, STFT/FBank/MFCC extraction, CNN and ResNet18 definitions, training, evaluation, plots, and paper comparison. It does not implement STM or Transformer components.

Run cells from top to bottom. Raw ShipsEar audio and metadata stay outside GitHub.

## 1. Install dependencies and import libraries

Run this cell once in Colab or a fresh Jupyter environment.

In [ ]:
%pip install -q numpy pandas soundfile librosa scikit-learn matplotlib seaborn torch torchvision openpyxl

In [ ]:
from pathlib import Path
import os, json, random, warnings
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from torchvision.models import resnet18

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
print("PyTorch:", torch.__version__)

## 2. Configure paths

Set these paths to match your environment. In Colab, mount Google Drive first if the dataset is stored there.

In [ ]:
# Optional Google Colab Drive mount:
# from google.colab import drive
# drive.mount('/content/drive')

METADATA_PATH = "data/metadata.csv"  # CSV you prepare from the supplied metadata workbook
AUDIO_ROOT = "data/ShipsEar"         # folder containing original audio files
OUTPUT_DIR = Path("data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If your metadata is an Excel workbook, inspect it before creating the CSV:
# workbook = pd.ExcelFile("path/to/ShipsEar_metadata.xlsx")
# print("Sheets:", workbook.sheet_names)
# for sheet in workbook.sheet_names:
#     frame = pd.read_excel("path/to/ShipsEar_metadata.xlsx", sheet_name=sheet)
#     print("\\n", sheet, frame.shape, list(frame.columns))
#     display(frame.head())

## 3. Load and validate metadata

Required columns: `audio_path` and `class_label` (A–E). `recording_order` should contain the actual chronological order for Dataset B. Do not rely on alphabetical filenames unless the dataset documentation confirms that ordering.

In [ ]:
LABELS = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
LABEL_NAMES = list(LABELS)

def load_metadata(metadata_path, audio_root=None):
    frame = pd.read_csv(metadata_path)
    required = {"audio_path", "class_label"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Metadata CSV missing columns: {sorted(missing)}")
    frame["class_label"] = frame["class_label"].astype(str).str.strip().str.upper()
    invalid = sorted(set(frame.class_label) - set(LABELS))
    if invalid:
        raise ValueError(f"Unknown labels {invalid}; expected A-E.")
    root = Path(audio_root) if audio_root else Path(".")
    frame["resolved_path"] = frame.audio_path.map(
        lambda p: str(Path(p) if Path(p).is_absolute() else root / p)
    )
    absent = [p for p in frame.resolved_path if not Path(p).exists()]
    if absent:
        raise FileNotFoundError("Missing audio files (first 10): " + ", ".join(absent[:10]))
    if frame.resolved_path.duplicated().any():
        raise ValueError("Duplicate audio_path entries found in metadata.")
    if "recording_order" not in frame.columns:
        warnings.warn("recording_order is missing. Dataset B will use sorted paths as an unverified fallback.")
    elif frame.recording_order.isna().any():
        warnings.warn("Some recording_order values are missing. Dataset B will use sorted paths as an unverified fallback.")
    return frame

# Uncomment after setting METADATA_PATH and AUDIO_ROOT:
# metadata = load_metadata(METADATA_PATH, AUDIO_ROOT)
# display(metadata.head())
# print("Original recordings:", len(metadata))
# print(metadata.class_label.value_counts())

## 4. Prepare 5-second clips and create Dataset A and Dataset B

- Dataset A: random clip-level split, 70:15:15.
- Dataset B: sort original recordings chronologically; first 70% of recordings go to training, remaining recordings are divided between validation and test. Clips from one original recording stay in one split.

This follows the split descriptions, but exact sample counts depend on the supplied metadata and recordings.

In [ ]:
def split_recordings_by_time(frame):
    if "recording_order" in frame.columns and frame.recording_order.notna().all():
        ordered = frame.sort_values("recording_order", kind="stable")
    else:
        warnings.warn("Using sorted audio_path as Dataset B chronology fallback. Verify this against the dataset documentation.")
        ordered = frame.sort_values("audio_path", kind="stable")
    paths = ordered.resolved_path.tolist()
    n_train = int(np.floor(0.70 * len(paths)))
    remaining = paths[n_train:]
    n_val = len(remaining) // 2
    return set(paths[:n_train]), set(remaining[:n_val]), set(remaining[n_val:])

def prepare_clips(metadata_path, audio_root, output_dir, sample_seconds=5.0, target_sr=16000, seed=42):
    meta = load_metadata(metadata_path, audio_root)
    output_dir = Path(output_dir)
    clips_dir = output_dir / "clips"
    clips_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    clip_length = int(sample_seconds * target_sr)

    for row in meta.itertuples(index=False):
        audio, sr = sf.read(row.resolved_path, dtype="float32", always_2d=False)
        if audio.ndim == 2:
            audio = audio.mean(axis=1)
        if sr != target_sr:
            audio = librosa.resample(audio, orig_sr=sr, target_sr=target_sr)
        usable = len(audio) // clip_length
        for clip_idx in range(usable):
            clip = audio[clip_idx * clip_length:(clip_idx + 1) * clip_length]
            if not np.isfinite(clip).all() or np.max(np.abs(clip)) < 1e-7:
                continue
            stem = Path(row.resolved_path).stem
            clip_path = clips_dir / f"{stem}_{clip_idx:05d}.wav"
            sf.write(str(clip_path), clip, target_sr)
            rows.append({
                "clip_path": str(clip_path),
                "source_path": str(row.resolved_path),
                "clip_index": clip_idx,
                "class_label": row.class_label,
                "label": LABELS[row.class_label],
                "recording_order": getattr(row, "recording_order", "")
            })

    clips = pd.DataFrame(rows)
    if clips.empty:
        raise RuntimeError("No usable 5-second clips created. Check audio paths and recordings.")

    train_idx, temp_idx = train_test_split(
        np.arange(len(clips)), test_size=0.30, random_state=seed,
        stratify=clips.label if clips.label.value_counts().min() >= 2 else None
    )
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=seed,
        stratify=clips.iloc[temp_idx].label if clips.iloc[temp_idx].label.value_counts().min() >= 2 else None
    )
    clips["split_A"] = ""
    clips.loc[train_idx, "split_A"] = "train"
    clips.loc[val_idx, "split_A"] = "val"
    clips.loc[test_idx, "split_A"] = "test"

    train_files, val_files, test_files = split_recordings_by_time(meta)
    clips["split_B"] = np.where(clips.source_path.isin(train_files), "train",
                         np.where(clips.source_path.isin(val_files), "val", "test"))
    clips.to_csv(output_dir / "clips_manifest.csv", index=False)
    clips.groupby(["split_A", "class_label"]).size().unstack(fill_value=0).to_csv(output_dir / "dataset_A_counts.csv")
    clips.groupby(["split_B", "class_label"]).size().unstack(fill_value=0).to_csv(output_dir / "dataset_B_counts.csv")
    print(f"Created {len(clips)} clips from {len(meta)} recordings.")
    print("\\nDataset A:", clips.split_A.value_counts().to_dict())
    print("Dataset B:", clips.split_B.value_counts().to_dict())
    return clips

# Run once:
# manifest = prepare_clips(METADATA_PATH, AUDIO_ROOT, OUTPUT_DIR, sample_seconds=5, target_sr=16000, seed=SEED)
# display(manifest.head())

## 5. Extract STFT, FBank and MFCC features

Feature settings used here: 254-point FFT, Hann window, 10 ms hop, and 128 mel filters for FBank/MFCC. MFCC uses 40 coefficients as an implementation choice because the paper does not specify this value.

In [ ]:
def extract_feature(audio_path, feature="fbank", sample_rate=16000,
                    n_fft=254, hop_ms=10, n_mels=128, n_mfcc=40):
    audio, sr = sf.read(str(audio_path), dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    if sr != sample_rate:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=sample_rate)
    hop = max(1, int(sample_rate * hop_ms / 1000))
    window = np.hanning(n_fft).astype(np.float32)

    if feature == "stft":
        spec = np.abs(librosa.stft(audio, n_fft=n_fft, hop_length=hop,
                                   win_length=n_fft, window=window, center=True))
    elif feature in ("fbank", "mfcc"):
        mel = librosa.feature.melspectrogram(
            y=audio, sr=sample_rate, n_fft=n_fft, hop_length=hop,
            win_length=n_fft, window=window, n_mels=n_mels, power=2.0
        )
        log_mel = librosa.power_to_db(mel, ref=np.max)
        spec = log_mel if feature == "fbank" else librosa.feature.mfcc(S=log_mel, n_mfcc=n_mfcc)
    else:
        raise ValueError("feature must be stft, fbank, or mfcc")

    spec = np.asarray(spec, dtype=np.float32)
    return (spec - spec.mean()) / (spec.std() + 1e-6)

def cache_features(manifest, output_dir, feature):
    output_dir = Path(output_dir)
    feature_dir = output_dir / feature
    feature_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for i, row in manifest.iterrows():
        destination = feature_dir / f"clip_{i:05d}.npy"
        if not destination.exists():
            np.save(destination, extract_feature(row.clip_path, feature=feature))
        paths.append(str(destination))
    result = manifest.copy()
    result["feature_path"] = paths
    manifest_path = output_dir / f"manifest_{feature}.csv"
    result.to_csv(manifest_path, index=False)
    print(f"Cached {len(paths)} {feature} arrays.")
    return result

# Run after preparing clips:
# feature_manifests = {name: cache_features(manifest, OUTPUT_DIR, name)
#                      for name in ["stft", "fbank", "mfcc"]}

## 6. Define CNN and ResNet18

CNN follows the two-convolution baseline: 1→64 and 64→256 channels, 5×5 kernels, Tanh, max-pooling, dropout 0.1, and two fully connected layers. The adaptive pooling and dense dimensions are implementation choices because the paper does not fully specify them. ResNet18 uses a single-channel first convolution.

In [ ]:
class PaperCNN(nn.Module):
    def __init__(self, num_classes=5, dropout=0.1):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, stride=1, padding=2),
            nn.Tanh(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 256, kernel_size=5, stride=1, padding=2),
            nn.Tanh(),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((8, 8))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(256 * 8 * 8, 512),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        return self.classifier(self.features(x))

def build_model(name="resnet18", num_classes=5):
    if name == "cnn":
        return PaperCNN(num_classes)
    if name == "resnet18":
        model = resnet18(weights=None)
        model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    raise ValueError("model must be cnn or resnet18")

print(build_model("cnn"))
print(build_model("resnet18"))

## 7. Dataset and training/evaluation functions

Training defaults: batch size 24, 100 epochs, CrossEntropyLoss, Adam, learning rate 1e-4. The best validation-accuracy checkpoint is restored before test evaluation.

In [ ]:
class FeatureDataset(Dataset):
    def __init__(self, frame, split_column, split):
        self.frame = frame[frame[split_column] == split].reset_index(drop=True)
        if self.frame.empty:
            raise ValueError(f"No samples for {split_column}={split}")
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        array = np.load(row.feature_path).astype(np.float32)
        x = torch.from_numpy(array).unsqueeze(0).unsqueeze(0)
        x = torch.nn.functional.interpolate(
            x, size=(128, 128), mode="bilinear", align_corners=False
        ).squeeze(0)
        return x, int(row.label)

def make_loader(frame, split_column, split, batch_size, shuffle):
    return DataLoader(FeatureDataset(frame, split_column, split),
                      batch_size=batch_size, shuffle=shuffle, num_workers=0)

def train_experiment(frame, dataset="A", feature="fbank", model_name="resnet18",
                     output_dir="outputs", epochs=100, batch_size=24, lr=1e-4, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    split_column = "split_A" if dataset.upper() == "A" else "split_B"
    output_path = Path(output_dir) / f"dataset_{dataset.upper()}" / feature / model_name
    output_path.mkdir(parents=True, exist_ok=True)

    train_loader = make_loader(frame, split_column, "train", batch_size, True)
    val_loader = make_loader(frame, split_column, "val", batch_size, False)
    test_loader = make_loader(frame, split_column, "test", batch_size, False)
    model = build_model(model_name).to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    best_accuracy = -1.0
    checkpoint_path = output_path / "best_model.pt"

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y)
        train_loss = total_loss / len(train_loader.dataset)

        model.eval()
        val_loss, predictions, targets = 0.0, [], []
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(DEVICE), y.to(DEVICE)
                logits = model(x)
                val_loss += criterion(logits, y).item() * len(y)
                predictions.extend(logits.argmax(1).cpu().tolist())
                targets.extend(y.cpu().tolist())
        val_loss /= len(val_loader.dataset)
        val_accuracy = accuracy_score(targets, predictions)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(float(val_accuracy))

        if val_accuracy > best_accuracy:
            best_accuracy = float(val_accuracy)
            torch.save({"model_state_dict": model.state_dict(), "epoch": epoch + 1,
                        "val_accuracy": best_accuracy, "model": model_name,
                        "dataset": dataset, "feature": feature}, checkpoint_path)
        print(f"Epoch {epoch+1:03d}/{epochs} | train_loss={train_loss:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_accuracy:.4f}")

    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    predictions, targets = [], []
    with torch.no_grad():
        for x, y in test_loader:
            logits = model(x.to(DEVICE))
            predictions.extend(logits.argmax(1).cpu().tolist())
            targets.extend(y.tolist())

    accuracy = float(accuracy_score(targets, predictions))
    report = classification_report(targets, predictions, labels=[0,1,2,3,4],
        target_names=LABEL_NAMES, output_dict=True, zero_division=0)
    matrix = confusion_matrix(targets, predictions, labels=[0,1,2,3,4])
    result = {"dataset": dataset, "feature": feature, "model": model_name,
              "test_accuracy": accuracy, "best_validation_accuracy": best_accuracy,
              "best_epoch": checkpoint["epoch"], "test_samples": len(targets),
              "classification_report": report, "confusion_matrix": matrix.tolist(),
              "device": str(DEVICE)}
    (output_path / "metrics.json").write_text(json.dumps(result, indent=2))
    (output_path / "history.json").write_text(json.dumps(history, indent=2))

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history["train_loss"], label="Train")
    axes[0].plot(history["val_loss"], label="Validation")
    axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[1].plot(history["val_accuracy"], label="Validation accuracy")
    axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    fig.tight_layout()
    fig.savefig(output_path / "training_curves.png", dpi=160)
    plt.show()
    plt.close(fig)

    plt.figure(figsize=(6, 5))
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=LABEL_NAMES, yticklabels=LABEL_NAMES)
    plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test confusion matrix")
    plt.tight_layout()
    plt.savefig(output_path / "confusion_matrix.png", dpi=160)
    plt.show()
    plt.close()

    print("Test accuracy:", f"{accuracy:.4%}")
    print(classification_report(targets, predictions, labels=[0,1,2,3,4],
                                target_names=LABEL_NAMES, zero_division=0))
    print("Saved artifacts to:", output_path)
    return result

## 8. Prepare features

Run this after creating the clips manifest. It extracts each feature type once and stores NumPy arrays and feature manifests.

In [ ]:
# manifest = pd.read_csv(OUTPUT_DIR / "clips_manifest.csv")
# feature_manifests = {}
# for feature_name in ["stft", "fbank", "mfcc"]:
#     feature_manifests[feature_name] = cache_features(manifest, OUTPUT_DIR, feature_name)
# print({k: len(v) for k, v in feature_manifests.items()})

## 9. Run experiments

There are 2 datasets × 3 feature types × 2 models = 12 experiments. First run a small smoke test (for example 2 epochs) to check the pipeline. Then set 100 epochs for the full runs. Training all 12 experiments can take substantial time.

In [ ]:
# Uncomment after feature extraction is complete.
# EPOCHS = 100
# BATCH_SIZE = 24
# LEARNING_RATE = 1e-4
# results = []
# for dataset_name in ["A", "B"]:
#     for feature_name in ["stft", "fbank", "mfcc"]:
#         feature_frame = pd.read_csv(OUTPUT_DIR / f"manifest_{feature_name}.csv")
#         for model_name in ["cnn", "resnet18"]:
#             print(f"\\n=== Dataset {dataset_name} | {feature_name} | {model_name} ===")
#             result = train_experiment(
#                 feature_frame, dataset=dataset_name, feature=feature_name,
#                 model_name=model_name, output_dir="outputs",
#                 epochs=EPOCHS, batch_size=BATCH_SIZE, lr=LEARNING_RATE, seed=SEED
#             )
#             results.append(result)

## 10. Summarize experiment results

This table contains results generated by your own runs. It must not be confused with the published paper's values.

In [ ]:
# if results:
#     summary = pd.DataFrame([
#         {k: r[k] for k in ["dataset", "feature", "model", "test_accuracy",
#                            "best_validation_accuracy", "best_epoch", "test_samples"]}
#         for r in results
#     ])
#     display(summary.sort_values(["dataset", "model", "feature"]))
#     summary.to_csv("outputs/experiment_summary.csv", index=False)

## 11. Published paper reference values

These are reference values reported in the paper, **not results from this notebook**.

| Experiment | Dataset A | Dataset B |
|---|---:|---:|
| ResNet18 + STFT (Table 2) | 76.9% | 60.1% |
| ResNet18 + FBank (Table 2) | 93.8% | 88.2% |
| ResNet18 + MFCC (Table 2) | 93.8% | 85.5% |
| CNN baseline (Table 4) | 84.0% | 39.9% |
| ResNet18 baseline (Table 4) | 93.7% | 88.1% |

Table 2 compares feature types using ResNet18; Table 4 compares model baselines. Compare your own results only after experiments finish. Explain differences through metadata mapping, split reconstruction, feature settings, random seed, architecture details not fully specified by the paper, and software/hardware differences.

## 12. Reproducibility and limitations

- The original ShipsEar recordings are not included in this repository.
- Dataset B is only meaningful when `recording_order` reflects the actual chronology of original recordings.
- The 5-second clip count and split counts depend on the supplied audio and metadata; do not claim exact paper counts until verified.
- MFCC coefficient count (40), per-clip standardization, resizing to 128×128, and CNN dense dimensions are implementation choices where the paper is not fully explicit.
- No accuracy should be reported until the notebook has actually been run on the dataset.

**Reference:** Li et al. (2022), “STM: Spectrogram Transformer Model for Underwater Acoustic Target Recognition,” *Journal of Marine Science and Engineering*, 10(10), 1428. https://doi.org/10.3390/jmse10101428